# 05c Ship Calibrated

Ship the calibration artifacts written by `05b_calibrate_wshed.ipynb`. This notebook does not rescore Wflow against USGS; it only reads the reviewed calibration summary/patch and stamps downstream catalog artifacts.

## Runtime

In [1]:
from pathlib import Path
import json

import pandas as pd
import yaml
from IPython.display import display

location_root = Path("..").resolve()

location_name = location_root.name
catalog_dir = location_root / "data/event_catalog/catalog"
scenario_catalog_path = catalog_dir / "scenario_catalog.csv"
calibration_root = location_root / "data/wflow/calibration"
calibration_summary_path = calibration_root / "usgs_wflow_calibration_summary.csv"
patch_json_path = calibration_root / "wflow_calibration_patch.json"
patch_yaml_path = calibration_root / "wflow_calibration_patch.yaml"

pd.set_option("display.max_colwidth", 160)
display(pd.Series({"location_root": str(location_root), "scenario_catalog": str(scenario_catalog_path)}))

location_root                                                       /home/grahamhults/projects/Flood-RM/locations/greensboro
scenario_catalog    /home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/scenario_catalog.csv
dtype: str

## Rerun Control


In [2]:
rerun = True

## Load 05b Calibration Artifacts

In [3]:
if not calibration_summary_path.exists():
    raise FileNotFoundError(f"Run 05b_calibrate_wshed.ipynb first; missing {calibration_summary_path}")
if patch_json_path.exists():
    calibration_patch = json.loads(patch_json_path.read_text(encoding="utf-8"))
elif patch_yaml_path.exists():
    calibration_patch = yaml.safe_load(patch_yaml_path.read_text(encoding="utf-8")) or {}
else:
    raise FileNotFoundError(f"Run 05b_calibrate_wshed.ipynb first; missing {patch_json_path} or {patch_yaml_path}")

calibration_summary = pd.read_csv(calibration_summary_path, dtype={"event_id": str, "site_no": str})
display(pd.Series({
    "calibration_status": calibration_patch.get("status"),
    "primary_reference_gage": calibration_patch.get("primary_reference_gage"),
    "k_calibration": calibration_patch.get("k_calibration"),
    "event_count": calibration_patch.get("event_count"),
    "summary_rows": len(calibration_summary),
    "summary_csv": str(calibration_summary_path),
    "patch_json": str(patch_json_path) if patch_json_path.exists() else "not_written",
}, name="loaded_wflow_calibration"))
display(calibration_summary.head(20))


calibration_status                                                                                                             ready_to_ship
primary_reference_gage                                                                                                              02094500
k_calibration                                                                                                                       0.737571
event_count                                                                                                                                2
summary_rows                                                                                                                              14
summary_csv               /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/calibration/usgs_wflow_calibration_summary.csv
patch_json                      /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/calibration/wflow_calibration_patch.json
Name: loaded_

,event_id,site_no,q_column,n,nse,kge,peak_bias_fraction,volume_bias_fraction
0,historical_0001_20180917T030000,02093800,Q_7,120,0.020670,-0.126454,0.973278,0.697415
1,historical_0001_20180917T030000,02093877,Q_8,120,-2.545387,-0.652281,2.378316,0.750260
2,historical_0001_20180917T030000,0209399200,Q_9,120,-3.116571,-1.128186,2.327918,1.084265
3,historical_0001_20180917T030000,02094500,Q_10,120,-4.001886,-0.665370,2.414456,0.601795
4,historical_0001_20180917T030000,02094659,Q_11,120,-0.361676,-0.054472,0.409681,0.855309
5,historical_0001_20180917T030000,02094770,Q_12,120,0.113037,-0.055159,0.904434,0.796112
6,historical_0001_20180917T030000,02094775,Q_13,120,0.504833,0.518515,0.075239,0.398572
7,historical_0001_20180917T030000,02095000,Q_14,120,0.011445,0.018007,1.257380,0.627405
8,historical_0001_20180917T030000,02095181,Q_15,120,-0.346315,0.119481,0.114024,0.691267
9,historical_0001_20180917T030000,02095271,Q_16,120,0.561204,0.666884,-0.072123,0.246676


## Ship Catalog

In [4]:
SHIP_CALIBRATED_CATALOG = True

scenario_catalog = pd.read_csv(scenario_catalog_path, dtype={"event_id": str})
calibrated_catalog = scenario_catalog.copy()
calibrated_catalog["wflow_calibration_status"] = calibration_patch.get("status")
calibrated_catalog["wflow_k_calibration"] = calibration_patch.get("k_calibration")
calibrated_catalog["wflow_calibration_reference_gage"] = calibration_patch.get("primary_reference_gage")
calibrated_catalog["wflow_calibration_patch"] = str(patch_yaml_path.relative_to(location_root))
calibrated_catalog["wflow_calibration_summary"] = str(calibration_summary_path.relative_to(location_root))

calibrated_catalog_path = catalog_dir / "scenario_catalog.calibrated.csv"
if SHIP_CALIBRATED_CATALOG:
    calibrated_catalog.to_csv(calibrated_catalog_path, index=False)

canonical_backup_path = catalog_dir / "scenario_catalog.pre_calibration_backup.csv"
if rerun:
    if not canonical_backup_path.exists():
        scenario_catalog.to_csv(canonical_backup_path, index=False)
    calibrated_catalog.to_csv(scenario_catalog_path, index=False)

display(pd.Series({
    "ship_calibrated_catalog": SHIP_CALIBRATED_CATALOG,
    "rerun": rerun,
    "calibrated_catalog": str(calibrated_catalog_path),
    "canonical_catalog": str(scenario_catalog_path),
    "canonical_backup": str(canonical_backup_path) if rerun else "not_written",
}, name="shipped_wflow_calibration"))
calibrated_catalog.head()


ship_calibrated_catalog                                                                                                                               True
rerun                                                                                                                                                 True
calibrated_catalog                     /home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/scenario_catalog.calibrated.csv
canonical_catalog                                 /home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/scenario_catalog.csv
canonical_backup           /home/grahamhults/projects/Flood-RM/locations/greensboro/data/event_catalog/catalog/scenario_catalog.pre_calibration_backup.csv
Name: shipped_wflow_calibration, dtype: object

,event_id,rainfall_mm,sample_rp_years,severity_band,sampling_weight,probability_weight,catalog_role,scenario_name,rainfall_template_member_id,rainfall_template_value,...,sampling_scheme,event_set,selection_role,event_origin,selection_reason,wflow_calibration_status,wflow_k_calibration,wflow_calibration_reference_gage,wflow_calibration_patch,wflow_calibration_summary
0,design_0000,51.379761,0.210306,mild,0.055497,0.036054,inland_design,base,rainfall_greensboro_72h_rank0467,51.866667,...,band_stratified_importance_rainfall,resilience_stress_training,resilience_stress_training,NaN,NaN,ready_to_ship,0.737571,02094500,data/wflow/calibration/wflow_calibration_patch.yaml,data/wflow/calibration/usgs_wflow_calibration_summary.csv
1,design_0001,53.416174,0.228664,mild,0.055497,0.036054,inland_design,base,rainfall_greensboro_72h_rank0427,53.577154,...,band_stratified_importance_rainfall,resilience_stress_training,resilience_stress_training,NaN,NaN,ready_to_ship,0.737571,02094500,data/wflow/calibration/wflow_calibration_patch.yaml,data/wflow/calibration/usgs_wflow_calibration_summary.csv
2,design_0002,54.088894,0.235047,mild,0.055497,0.036054,inland_design,base,rainfall_greensboro_72h_rank0421,54.226593,...,band_stratified_importance_rainfall,resilience_stress_training,resilience_stress_training,NaN,NaN,ready_to_ship,0.737571,02094500,data/wflow/calibration/wflow_calibration_patch.yaml,data/wflow/calibration/usgs_wflow_calibration_summary.csv
3,design_0003,55.332571,0.247284,mild,0.055497,0.036054,inland_design,base,rainfall_greensboro_72h_rank0402,55.325844,...,band_stratified_importance_rainfall,resilience_stress_training,resilience_stress_training,NaN,NaN,ready_to_ship,0.737571,02094500,data/wflow/calibration/wflow_calibration_patch.yaml,data/wflow/calibration/usgs_wflow_calibration_summary.csv
4,design_0004,56.974110,0.264338,mild,0.055497,0.036054,inland_design,base,rainfall_greensboro_72h_rank0364,57.940638,...,band_stratified_importance_rainfall,resilience_stress_training,resilience_stress_training,NaN,NaN,ready_to_ship,0.737571,02094500,data/wflow/calibration/wflow_calibration_patch.yaml,data/wflow/calibration/usgs_wflow_calibration_summary.csv
